# Part 5: More replicas, and where each request goes

Fill both routers first (`pytest tests/test_routing.py`). The runs, from the repo root, one at a time:

```bash
# how many replicas: Sarathi on 2 replicas, round-robin (about 4 minutes)
REPLICAS=2 GLOBAL=round_robin SCHED=sarathi CHUNK=512 QPS=5.53 bash scripts/run_default.sh traces/azure_conversation.csv

# routers on the long-short bursts trace (seconds each)
for G in round_robin lor lot; do
  REPLICAS=2 GLOBAL=$G SCHED=sarathi CHUNK=512 QPS=8 bash scripts/run_default.sh traces/long_short_bursts.csv
done

# LOT on the real trace (about 4 minutes)
REPLICAS=2 GLOBAL=lot SCHED=sarathi CHUNK=512 QPS=5.53 bash scripts/run_default.sh traces/azure_conversation.csv
```

In [ ]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "traces").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analysis import workload as wl
from analysis.replay import busy_percent, describe_run, load_run
from analysis.workload import load_trace, predict_queue_wait

azure = load_trace(ROOT / "traces" / "azure_conversation.csv")
bursts = load_trace(ROOT / "traces" / "long_short_bursts.csv")

def summary(run_dir, trace):
    run = load_run(run_dir, trace)
    tbt = run["tbt"].set_index("percentile").seconds
    req = run["requests"]
    return {"wait_p99": req.wait.quantile(0.99), "e2e_p99": req.e2e.quantile(0.99),
            "tbt_p99_ms": tbt[99] * 1e3, "busy": busy_percent(run_dir)}

## How many replicas?

In [ ]:
for n in (1, 2, 3):
    w = predict_queue_wait(azure, prefill_tok_per_s=n * wl.PREFILL_TOK_PER_S,
                           decode_tok_per_s=n * wl.DECODE_TOK_PER_S)
    print(f"{n} replica(s): predicted P99 wait {np.quantile(w, 0.99):.2f} s")

ONE_REPLICA = ROOT / "simulator_output" / "REPLACE_PART3_SARATHI_C512"   # from Part 3
TWO_RR = ROOT / "simulator_output" / "REPLACE_AZURE_2_ROUND_ROBIN"
for d in (ONE_REPLICA, TWO_RR):
    s = summary(d, azure)
    print(f"{describe_run(d):42s} P99 wait {s['wait_p99']:6.2f} s   P99 time between tokens {s['tbt_p99_ms']:.0f} ms")

## Routers on the long-short bursts trace

In [ ]:
BURST_RUNS = [ROOT / "simulator_output" / d for d in [
    "REPLACE_BURSTS_ROUND_ROBIN", "REPLACE_BURSTS_LOR", "REPLACE_BURSTS_LOT",
]]
for d in BURST_RUNS:
    s = summary(d, bursts)
    print(f"{describe_run(d):42s} wait P99 {s['wait_p99']:5.2f} s   end-to-end P99 {s['e2e_p99']:5.2f} s   "
          f"busy % {' / '.join(f'{b:.0f}' for b in s['busy'])}")

## Round-robin vs LOT on the real trace

In [ ]:
TWO_LOT = ROOT / "simulator_output" / "REPLACE_AZURE_2_LOT"
for d in (TWO_RR, TWO_LOT):
    s = summary(d, azure)
    print(f"{describe_run(d):42s} P99 wait {s['wait_p99']:5.2f} s   P99 time between tokens {s['tbt_p99_ms']:.0f} ms")

## Questions

**Q1.** Fill in the table. What is the smallest number of replicas that meets both
targets (P99 wait under 2 s, P99 time between tokens under 100 ms)? How
close was your Part 1 model to Vidur?

| Replicas | Predicted P99 wait (Part 1 model) | Vidur P99 wait | Vidur P99 time between tokens |
|---|---|---|---|
| 1 | | (Part 3, C = 512) | |
| 2 | | | |
| 3 | | — | — |

**Your answer:**

**Q2.** Fill in the table for the long-short bursts trace. Why does round-robin leave one
replica almost idle? Why does LOT beat LOR? Use the worked example.

| Router | Wait P99 | End-to-end P99 | Busy % (replica 1 / 2) |
|---|---|---|---|
| Round-robin | | | |
| LOR | | | |
| LOT | | | |

**Your answer:**

**Q3.** Vidur's original LOR counted only requests **waiting** in each replica's
queue, not running ones. On the real trace with 2 replicas, those queues are
almost always empty. Which replica would that router pick for almost every
request, and why? Why is counting running requests the right definition?

**Your answer:**

**Q4.** On the real trace with 2 replicas, compare round-robin with LOT: P99 wait
and P99 time between tokens. Does the router matter here? In one sentence:
when is a load-aware router worth it?

**Your answer:**